# 🧹 Limpieza de Datos con Pandas — Manual completo

**Para:** Borja
**Enfoque:** todas las herramientas de limpieza de Pandas, sobre un único
dataset "sucio" realista (un export de CRM con los problemas típicos que te
vas a encontrar en cualquier trabajo real).

**Principio que vertebra todo el notebook:** *garbage in, garbage out* — si
los datos de entrada están sucios, cualquier análisis o modelo construido
encima será, como mucho, tan bueno como esos datos. La limpieza no es un
paso previo aburrido: es donde se decide si el análisis final va a ser
fiable o no.

**Índice de problemas que vas a saber resolver:**

1. Diagnóstico inicial: detectar qué está mal antes de tocar nada.
2. Duplicados (exactos y "lógicos").
3. Valores nulos: detectar, eliminar, rellenar.
4. Texto inconsistente: mayúsculas, espacios, erratas.
5. Tipos de datos incorrectos: números como texto, fechas en formatos mixtos.
6. Valores fuera de rango lógico (outliers de negocio).
7. Validación de formato con expresiones regulares.
8. Un pipeline completo de limpieza, de principio a fin, nivel profesional.


## 1. El dataset: un export de CRM con problemas reales

Antes de explicar nada, vamos a crear el dataset que usaremos en todo el
notebook. Tiene, a propósito, prácticamente todos los problemas que te vas a
encontrar en un dataset real: duplicados, nulos, texto con mayúsculas y
espacios inconsistentes, una errata de ciudad, números guardados como texto
con símbolos de moneda, fechas en dos formatos distintos (e incluso una fecha
imposible), edades fuera de rango, un importe negativo, y emails mal
escritos.


In [1]:
import pandas as pd
import numpy as np

datos = [
    {"id_cliente": 1, "nombre": "Ana García", "ciudad": "Madrid", "edad": 34,
     "email": "ana.garcia@correo.com", "ingresos_anuales": "45000",
     "fecha_registro": "2023-05-12", "importe_compra": 320.50, "producto": "Mesa"},
    {"id_cliente": 1, "nombre": "Ana García", "ciudad": "Madrid", "edad": 34,
     "email": "ana.garcia@correo.com", "ingresos_anuales": "45000",
     "fecha_registro": "2023-05-12", "importe_compra": 320.50, "producto": "Mesa"},
    {"id_cliente": 2, "nombre": "  Luis Pérez", "ciudad": "BARCELONA", "edad": 28,
     "email": "luis.perez@correo.com", "ingresos_anuales": "38.500€",
     "fecha_registro": "12/06/2023", "importe_compra": 150.00, "producto": "silla"},
    {"id_cliente": 3, "nombre": "Eva Martín ", "ciudad": "madrid ", "edad": -5,
     "email": "eva.martin@correo.com", "ingresos_anuales": "$52,000",
     "fecha_registro": "2023-07-01", "importe_compra": 89.99, "producto": "SILLA"},
    {"id_cliente": 4, "nombre": "Iván Soto", "ciudad": "Valencia", "edad": 41,
     "email": "ivan_sin_arroba.com", "ingresos_anuales": np.nan,
     "fecha_registro": "20/08/2023", "importe_compra": 410.00, "producto": "Armario"},
    {"id_cliente": 5, "nombre": "María López", "ciudad": "Vlencia", "edad": 200,
     "email": "maria.lopez@correo.com", "ingresos_anuales": "41000",
     "fecha_registro": "2023-09-15", "importe_compra": 275.30, "producto": "Mesa "},
    {"id_cliente": 6, "nombre": "Carlos Ruiz", "ciudad": "Sevilla", "edad": np.nan,
     "email": "carlos.ruiz@correo.com", "ingresos_anuales": "47.200€",
     "fecha_registro": "2023-10-02", "importe_compra": -50.00, "producto": "Lampara"},
    {"id_cliente": 7, "nombre": "Lucía Vidal", "ciudad": "SEVILLA", "edad": 31,
     "email": np.nan, "ingresos_anuales": "39000",
     "fecha_registro": "31/02/2023", "importe_compra": 198.40, "producto": "Estanteria"},
    {"id_cliente": 8, "nombre": "Pablo Núñez", "ciudad": "Barcelona ", "edad": 27,
     "email": "pablo.nunez@correo.com", "ingresos_anuales": "36,000",
     "fecha_registro": "2023-11-20", "importe_compra": 320.50, "producto": "Mesa"},
    {"id_cliente": 9, "nombre": "Sara Cano", "ciudad": "Madrid", "edad": 39,
     "email": "sara.cano@correo.com", "ingresos_anuales": "61000",
     "fecha_registro": "2023-12-01", "importe_compra": 540.00, "producto": "ARMARIO "},
    {"id_cliente": 5, "nombre": "María López", "ciudad": "Valencia", "edad": 35,
     "email": "maria.lopez@correo.com", "ingresos_anuales": "41000",
     "fecha_registro": "2023-09-15", "importe_compra": 275.30, "producto": "Mesa"},
]

clientes_crm = pd.DataFrame(datos)
print("Forma:", clientes_crm.shape)
clientes_crm


Forma: (11, 9)


,id_cliente,nombre,ciudad,edad,email,ingresos_anuales,fecha_registro,importe_compra,producto
0,1,Ana García,Madrid,34.0,ana.garcia@correo.com,45000,2023-05-12,320.50,Mesa
1,1,Ana García,Madrid,34.0,ana.garcia@correo.com,45000,2023-05-12,320.50,Mesa
2,2,Luis Pérez,BARCELONA,28.0,luis.perez@correo.com,38.500€,12/06/2023,150.00,silla
3,3,Eva Martín,madrid,-5.0,eva.martin@correo.com,"$52,000",2023-07-01,89.99,SILLA
4,4,Iván Soto,Valencia,41.0,ivan_sin_arroba.com,NaN,20/08/2023,410.00,Armario
5,5,María López,Vlencia,200.0,maria.lopez@correo.com,41000,2023-09-15,275.30,Mesa
6,6,Carlos Ruiz,Sevilla,NaN,carlos.ruiz@correo.com,47.200€,2023-10-02,-50.00,Lampara
7,7,Lucía Vidal,SEVILLA,31.0,NaN,39000,31/02/2023,198.40,Estanteria
8,8,Pablo Núñez,Barcelona,27.0,pablo.nunez@correo.com,"36,000",2023-11-20,320.50,Mesa
9,9,Sara Cano,Madrid,39.0,sara.cano@correo.com,61000,2023-12-01,540.00,ARMARIO


Tómate un momento para mirar la tabla: `BARCELONA`, `madrid ` (con espacio) y
`Madrid` son, en realidad, la misma ciudad escrita de 3 formas distintas.
`Vlencia` es una errata de `Valencia`. El cliente `5` aparece dos veces, con
datos algo distintos. Esto es exactamente el tipo de cosas que te vas a
encontrar al abrir un CSV real por primera vez.


## 2. Diagnóstico inicial: el chequeo médico del dataset

**Qué es:** el primer vistazo, siempre antes de tocar nada, para entender qué
problemas tiene el dataset y de qué tamaño son.

### 🎈 Analogía sencilla

Es como un chequeo médico antes de empezar un tratamiento: no operas a nadie
sin antes mirar los análisis. Aquí, antes de limpiar nada, miras cuántos
huecos hay, cuántas filas repetidas, y qué valores "raros" aparecen en las
columnas de texto.

### 🐍 Código Python


In [2]:
clientes_crm.info()


<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_cliente        11 non-null     int64  
 1   nombre            11 non-null     str    
 2   ciudad            11 non-null     str    
 3   edad              10 non-null     float64
 4   email             10 non-null     str    
 5   ingresos_anuales  10 non-null     str    
 6   fecha_registro    11 non-null     str    
 7   importe_compra    11 non-null     float64
 8   producto          11 non-null     str    
dtypes: float64(2), int64(1), str(6)
memory usage: 924.0 bytes


In [3]:
print("--- Nulos por columna ---")
print(clientes_crm.isna().sum())


--- Nulos por columna ---
id_cliente          0
nombre              0
ciudad              0
edad                1
email               1
ingresos_anuales    1
fecha_registro      0
importe_compra      0
producto            0
dtype: int64


In [4]:
print("--- Duplicados exactos ---")
print(clientes_crm.duplicated().sum())


--- Duplicados exactos ---
1


In [5]:
print("--- Valores únicos de 'ciudad' ---")
print(clientes_crm["ciudad"].unique())
print()
print("--- Valores únicos de 'producto' ---")
print(clientes_crm["producto"].unique())


--- Valores únicos de 'ciudad' ---
<StringArray>
[    'Madrid',  'BARCELONA',    'madrid ',   'Valencia',    'Vlencia',
    'Sevilla',    'SEVILLA', 'Barcelona ']
Length: 8, dtype: str

--- Valores únicos de 'producto' ---
<StringArray>
[      'Mesa',      'silla',      'SILLA',    'Armario',      'Mesa ',
    'Lampara', 'Estanteria',   'ARMARIO ']
Length: 8, dtype: str


In [6]:
print("--- value_counts() de 'ciudad' ---")
print(clientes_crm["ciudad"].value_counts())


--- value_counts() de 'ciudad' ---
ciudad
Madrid        3
Valencia      2
BARCELONA     1
madrid        1
Vlencia       1
Sevilla       1
SEVILLA       1
Barcelona     1
Name: count, dtype: int64


### 🔍 Explicación línea a línea

- `.info()` → filas, columnas, cuántos valores **no nulos** tiene cada una, y
  su tipo. Si una columna numérica aparece como `str`/`object`, ya sabes que
  hay un problema de tipo escondido.
- `.isna().sum()` → cuántos huecos hay, columna por columna.
- `.duplicated().sum()` → cuántas filas son copias exactas de otra.
- `.unique()` → todos los valores distintos que aparecen en una columna de
  texto. Es la forma más rápida de detectar que `"Madrid"`, `"MADRID"` y
  `"madrid "` están conviviendo como si fueran categorías distintas.
- `.value_counts()` → como `.unique()`, pero contando cuántas veces aparece
  cada valor — muy útil para detectar errores raros que aparecen solo 1 vez
  (normalmente, erratas) frente a las categorías "de verdad", que aparecen
  muchas veces.

### 💼 Aplicación profesional

Estos 5 comandos son, literalmente, lo primero que debería hacer cualquier
analista con cualquier dataset nuevo, **antes** de escribir ni una sola línea
de análisis. Saltarse este paso es la causa número uno de informes con
conclusiones erróneas por culpa de datos sucios sin detectar.

---

### 📝 Ejercicio 1 (básico) — Diagnóstico de la columna `producto`

Sobre `clientes_crm`, muestra los valores únicos de la columna `producto`, y
cuenta cuántos valores nulos tiene en total **todo** el DataFrame (un único
número, sumando todas las columnas).


In [7]:
# Ejercicio 1 — Tu código aquí


#### 💡 Solución — Ejercicio 1


In [8]:
print(clientes_crm["producto"].unique())
print("Nulos totales en el dataset:", clientes_crm.isna().sum().sum())


<StringArray>
[      'Mesa',      'silla',      'SILLA',    'Armario',      'Mesa ',
    'Lampara', 'Estanteria',   'ARMARIO ']
Length: 8, dtype: str
Nulos totales en el dataset: 3


> 💡 `.isna().sum()` te da los nulos **por columna**; encadenar otro
> `.sum()` suma esos resultados entre sí y te da el total general del
> DataFrame completo.


## 3. Duplicados: exactos y "lógicos"

**Qué es:** filas que representan la misma información repetida — a veces de
forma idéntica (duplicado exacto), a veces con pequeñas diferencias porque se
introdujeron en momentos distintos (duplicado "lógico", el mismo cliente con
algún dato distinto).

### 🐍 Código Python: duplicados exactos


In [9]:
print("Filas duplicadas (exactas):", clientes_crm.duplicated().sum())

sin_dup_exactos = clientes_crm.drop_duplicates()
print("Forma antes:", clientes_crm.shape, "- después:", sin_dup_exactos.shape)


Filas duplicadas (exactas): 1
Forma antes: (11, 9) - después: (10, 9)


### 🐍 Código Python: duplicados "lógicos" (mismo cliente, datos distintos)

Fíjate en el `id_cliente` 5: aparece dos veces, pero con `ciudad` y `edad`
distintas. No son duplicados exactos (`.duplicated()` no los detecta), pero
de cara al negocio, **son la misma persona** registrada dos veces.


In [10]:
print(sin_dup_exactos[sin_dup_exactos["id_cliente"].duplicated(keep=False)])


    id_cliente       nombre    ciudad   edad                   email  \
5            5  María López   Vlencia  200.0  maria.lopez@correo.com   
10           5  María López  Valencia   35.0  maria.lopez@correo.com   

   ingresos_anuales fecha_registro  importe_compra producto  
5             41000     2023-09-15           275.3    Mesa   
10            41000     2023-09-15           275.3     Mesa  


In [11]:
# subset: duplicado solo si coincide ESA columna en concreto
duplicados_logicos = sin_dup_exactos.duplicated(subset=["id_cliente"], keep=False)
print("Filas con id_cliente repetido:", duplicados_logicos.sum())

# keep="last": al eliminar, nos quedamos con la ÚLTIMA aparición de cada id_cliente
sin_dup_logicos = sin_dup_exactos.drop_duplicates(subset=["id_cliente"], keep="last")
print("Forma final:", sin_dup_logicos.shape)


Filas con id_cliente repetido: 2
Forma final: (9, 9)


### 🔍 Explicación línea a línea

- `duplicated()` sin argumentos → compara **todas** las columnas; solo marca
  `True` si la fila es idéntica en todo a otra anterior.
- `duplicated(subset=["id_cliente"], keep=False)` → ahora solo mira esa
  columna. `keep=False` marca **todas** las apariciones repetidas (no solo la
  segunda), útil para inspeccionarlas todas antes de decidir qué hacer.
- `drop_duplicates(subset=["id_cliente"], keep="last")` → al eliminar, decide
  qué copia conservar: `"first"` (la primera, por defecto), `"last"` (la
  última) o `False` (eliminar **todas** las copias, sin quedarte con
  ninguna).

### ⚠️ Error común

Usar `drop_duplicates()` sin `subset` cuando en realidad lo que quieres es
deduplicar por una clave de negocio (como `id_cliente`). Si las filas no son
**exactamente** iguales en todas las columnas, `drop_duplicates()` sin
`subset` no detectará el duplicado lógico, y te quedarás con información de
un mismo cliente repartida en dos filas distintas sin darte cuenta.

### 💼 Aplicación profesional

Los duplicados lógicos son extremadamente comunes en datos de clientes: la
misma persona se registra dos veces, un mismo pedido se importa dos veces
desde sistemas distintos, etc. Decidir **qué fila conservar** (`first`,
`last`, o un criterio de negocio más elaborado) es una decisión real que
toma un analista, no algo automático.

---

### 📝 Ejercicio 2 (básico-intermedio) — Limpiar ambos tipos de duplicado

Sobre `clientes_crm`, elimina primero los duplicados exactos, y después los
duplicados lógicos por `id_cliente` (quedándote con la **primera**
aparición esta vez, al contrario que en el ejemplo). Comprueba la forma final.


In [12]:
# Ejercicio 2 — Tu código aquí


#### 💡 Solución — Ejercicio 2


In [13]:
paso1 = clientes_crm.drop_duplicates()
paso2 = paso1.drop_duplicates(subset=["id_cliente"], keep="first")

print("Forma original:", clientes_crm.shape)
print("Forma final:", paso2.shape)


Forma original: (11, 9)
Forma final: (9, 9)


## 4. Valores nulos: detectar, eliminar, rellenar

**Qué es:** las tres operaciones básicas frente a un hueco (`NaN`) en los
datos: confirmarlo, quitar la fila/columna que lo tiene, o sustituirlo por
algún valor razonable.

### 🐍 Código Python: detectar


In [14]:
print(clientes_crm.isna().sum())
print()
print(clientes_crm[clientes_crm["email"].isna()][["nombre", "email"]])


id_cliente          0
nombre              0
ciudad              0
edad                1
email               1
ingresos_anuales    1
fecha_registro      0
importe_compra      0
producto            0
dtype: int64

        nombre email
7  Lucía Vidal   NaN


### 🐍 Código Python: eliminar con `dropna`


In [15]:
# how='any' (por defecto): elimina la fila si TIENE AL MENOS UN nulo en las columnas indicadas
sin_email_nulo = clientes_crm.dropna(subset=["email"])
print("Forma tras quitar filas sin email:", sin_email_nulo.shape)

# thresh: exige un número MÍNIMO de valores no nulos para conservar la fila
con_thresh = clientes_crm.dropna(thresh=8)   # se queda con filas que tengan al menos 8 columnas rellenas
print("Forma con thresh=8:", con_thresh.shape)


Forma tras quitar filas sin email: (10, 9)
Forma con thresh=8: (11, 9)


### 🐍 Código Python: rellenar con `fillna`


In [16]:
relleno_fijo = clientes_crm.copy()
relleno_fijo["email"] = relleno_fijo["email"].fillna("sin_email@desconocido.com")
print(relleno_fijo[["nombre", "email"]])


          nombre                      email
0     Ana García      ana.garcia@correo.com
1     Ana García      ana.garcia@correo.com
2     Luis Pérez      luis.perez@correo.com
3    Eva Martín       eva.martin@correo.com
4      Iván Soto        ivan_sin_arroba.com
5    María López     maria.lopez@correo.com
6    Carlos Ruiz     carlos.ruiz@correo.com
7    Lucía Vidal  sin_email@desconocido.com
8    Pablo Núñez     pablo.nunez@correo.com
9      Sara Cano       sara.cano@correo.com
10   María López     maria.lopez@correo.com


In [17]:
relleno_media = clientes_crm.copy()
relleno_media["edad"] = relleno_media["edad"].fillna(relleno_media["edad"].mean())
print(relleno_media[["nombre", "edad"]])


          nombre   edad
0     Ana García   34.0
1     Ana García   34.0
2     Luis Pérez   28.0
3    Eva Martín    -5.0
4      Iván Soto   41.0
5    María López  200.0
6    Carlos Ruiz   46.4
7    Lucía Vidal   31.0
8    Pablo Núñez   27.0
9      Sara Cano   39.0
10   María López   35.0


### 🔍 Explicación línea a línea

- `dropna(subset=[...])` → elimina filas con nulo **en esas columnas
  concretas**; `how="all"` (en vez del `"any"` por defecto) solo eliminaría si
  **todas** las columnas indicadas están vacías a la vez.
- `dropna(thresh=8)` → conserva la fila solo si tiene al menos `8` valores no
  nulos en total — útil cuando quieres descartar filas "demasiado vacías" sin
  fijarte en una columna concreta.
- `fillna(valor)` → sustituye cada `NaN` por el valor que le pases. Puede ser
  un valor fijo (`"sin_email@desconocido.com"`), o un cálculo (la media, la
  mediana, la moda de esa misma columna).
- También existen `fillna(method="ffill")` (rellena con el valor anterior,
  típico en series temporales) y `.interpolate()` (estima el valor que
  "encajaría" entre el anterior y el siguiente).

### ⚠️ Error común

Rellenar un nulo con la **media** de una columna que tiene valores
disparatados sin limpiar primero (como la `edad` con un `-5` y un `200` en
este dataset): la media saldría artificialmente distorsionada por esos
valores imposibles. El orden correcto casi siempre es: primero corregir
outliers, **después** decidir si rellenar o no los nulos restantes.

### 💼 Aplicación profesional

No existe una única respuesta correcta de "eliminar vs. rellenar": depende de
**cuántos** nulos hay (¿es el 2% de las filas, o el 60%?), de si esa columna
es crítica para el análisis, y del contexto de negocio. Un analista
experimentado siempre justifica esta decisión, en vez de aplicar
automáticamente la misma receta a todo.

---

### 📝 Ejercicio 3 (intermedio) — Rellenar de forma justificada

Sobre `clientes_crm`, rellena los nulos de `ingresos_anuales` con la
**mediana** de esa columna (ten en cuenta que está como texto — de momento
ignora ese problema, lo resolveremos en la siguiente sección; para este
ejercicio, convierte solo esa columna a número de forma rápida con
`pd.to_numeric(..., errors="coerce")` antes de calcular la mediana).


In [18]:
# Ejercicio 3 — Tu código aquí


#### 💡 Solución — Ejercicio 3


In [19]:
ingresos_numerico = pd.to_numeric(
    clientes_crm["ingresos_anuales"].str.replace(r"[^\d]", "", regex=True),
    errors="coerce",
)

mediana = ingresos_numerico.median()
ingresos_relleno = ingresos_numerico.fillna(mediana)

print("Mediana usada para rellenar:", mediana)
print(ingresos_relleno)


Mediana usada para rellenar: 43000.0
0     45000.0
1     45000.0
2     38500.0
3     52000.0
4     43000.0
5     41000.0
6     47200.0
7     39000.0
8     36000.0
9     61000.0
10    41000.0
Name: ingresos_anuales, dtype: float64


## 5. Texto inconsistente: mayúsculas, espacios y erratas

**Qué es:** normalizar columnas de texto para que la misma categoría no
aparezca escrita de varias formas distintas.

### 🐍 Código Python: espacios y mayúsculas


In [20]:
print("Antes:", clientes_crm["ciudad"].unique())

ciudad_normalizada = clientes_crm["ciudad"].str.strip().str.title()
print("Después de strip + title:", ciudad_normalizada.unique())


Antes: <StringArray>
[    'Madrid',  'BARCELONA',    'madrid ',   'Valencia',    'Vlencia',
    'Sevilla',    'SEVILLA', 'Barcelona ']
Length: 8, dtype: str
Después de strip + title: <StringArray>
['Madrid', 'Barcelona', 'Valencia', 'Vlencia', 'Sevilla']
Length: 5, dtype: str


### 🐍 Código Python: corregir erratas con `replace`


In [21]:
correcciones = {"Vlencia": "Valencia"}
ciudad_corregida = ciudad_normalizada.replace(correcciones)
print("Después de corregir la errata:", ciudad_corregida.unique())


Después de corregir la errata: <StringArray>
['Madrid', 'Barcelona', 'Valencia', 'Sevilla']
Length: 4, dtype: str


### 🔍 Explicación línea a línea

- `.str.strip()` → quita espacios sobrantes al principio/final del texto
  (`"madrid "` → `"madrid"`). No toca espacios en medio del texto.
- `.str.title()` → pone en mayúscula la primera letra de cada palabra
  (`"madrid"` → `"Madrid"`), normalizando may/min de un plumazo.
- `.replace({"viejo": "nuevo"})` sobre una columna → sustituye valores
  **exactos** completos, no fragmentos — es la herramienta para corregir
  erratas conocidas, una vez que ya las has detectado con `.unique()`.

### ⚠️ Error común

Hacer `.replace()` **antes** de `.strip()`/`.title()`: si todavía hay
`"Vlencia"`, `" Vlencia"` y `"VLENCIA"` conviviendo, tu diccionario de
correcciones tendría que cubrir las 3 variantes a mano. Conviene normalizar
primero may/min y espacios, y corregir erratas **después**, cuando ya solo
queda una variante por error real.

### 💼 Aplicación profesional

Esto es exactamente lo que evita que una tabla pivote o un `groupby` te
separe "Madrid" en 3 filas distintas cuando en realidad es una sola ciudad —
un error de limpieza de texto sin detectar puede hacer que un informe
muestre números "más repartidos" de lo que son en realidad.

---

### 📝 Ejercicio 4 (intermedio) — Normalizar `producto`

Normaliza la columna `producto` (espacios + mayúsculas) y comprueba que,
después de hacerlo, solo quedan 5 valores únicos (no 8, como al principio).


In [22]:
# Ejercicio 4 — Tu código aquí


#### 💡 Solución — Ejercicio 4


In [23]:
producto_normalizado = clientes_crm["producto"].str.strip().str.title()
print(producto_normalizado.unique())
print("Número de valores únicos:", producto_normalizado.nunique())


<StringArray>
['Mesa', 'Silla', 'Armario', 'Lampara', 'Estanteria']
Length: 5, dtype: str
Número de valores únicos: 5


## 6. Tipos de datos incorrectos: números y fechas guardados como texto

**Qué es:** convertir columnas que deberían ser numéricas o de fecha, pero
que llegaron como texto (a veces con símbolos, a veces en formatos
distintos).

### 🐍 Código Python: limpiar números con símbolos de moneda


In [24]:
print(clientes_crm["ingresos_anuales"].tolist())


['45000', '45000', '38.500€', '$52,000', nan, '41000', '47.200€', '39000', '36,000', '61000', '41000']


In [25]:
solo_digitos = clientes_crm["ingresos_anuales"].str.replace(r"[^\d]", "", regex=True)
ingresos_limpios = pd.to_numeric(solo_digitos, errors="coerce")

print(ingresos_limpios)


0     45000.0
1     45000.0
2     38500.0
3     52000.0
4         NaN
5     41000.0
6     47200.0
7     39000.0
8     36000.0
9     61000.0
10    41000.0
Name: ingresos_anuales, dtype: float64


### 🔍 Explicación línea a línea

- `.str.replace(r"[^\d]", "", regex=True)` → una expresión regular que dice
  "sustituye cualquier carácter que **no** sea un dígito (`\d`) por nada".
  Elimina de un plumazo `€`, `$`, comas y puntos usados como separador de
  miles, sin que te importe cuál de esos símbolos usaba cada fila.
- `pd.to_numeric(..., errors="coerce")` → convierte el texto resultante a
  número. `errors="coerce"` es la pieza clave: si algún valor no se puede
  convertir (por ejemplo, si quedó vacío), en vez de romper todo el programa
  con un error, pone `NaN` en esa posición y continúa con el resto.

### 🐍 Código Python: fechas en formatos mixtos


In [26]:
print(clientes_crm["fecha_registro"].tolist())


['2023-05-12', '2023-05-12', '12/06/2023', '2023-07-01', '20/08/2023', '2023-09-15', '2023-10-02', '31/02/2023', '2023-11-20', '2023-12-01', '2023-09-15']


In [27]:
es_formato_iso = clientes_crm["fecha_registro"].str.contains("-")

fechas_limpias = pd.Series(pd.NaT, index=clientes_crm.index, dtype="datetime64[ns]")
fechas_limpias[es_formato_iso] = pd.to_datetime(
    clientes_crm.loc[es_formato_iso, "fecha_registro"], format="%Y-%m-%d", errors="coerce"
)
fechas_limpias[~es_formato_iso] = pd.to_datetime(
    clientes_crm.loc[~es_formato_iso, "fecha_registro"], format="%d/%m/%Y", errors="coerce"
)

print(fechas_limpias)


0    2023-05-12
1    2023-05-12
2    2023-06-12
3    2023-07-01
4    2023-08-20
5    2023-09-15
6    2023-10-02
7           NaT
8    2023-11-20
9    2023-12-01
10   2023-09-15
dtype: datetime64[ns]


### 🔍 Explicación línea a línea

- `.str.contains("-")` → detecta qué filas usan el separador `-` (formato
  ISO `AAAA-MM-DD`) frente a las que usan `/` (formato europeo `DD/MM/AAAA`).
- Se convierte **cada grupo por separado**, indicándole a `pd.to_datetime` el
  `format` exacto que le corresponde a ese grupo — esto es más fiable que
  pedirle a Pandas que "adivine" el formato de fechas mezcladas, que puede
  confundir día y mes en casos ambiguos.
- `errors="coerce"` aquí también: la fecha `"31/02/2023"` (un 31 de febrero,
  que no existe en ningún calendario) se convierte en `NaT` (*Not a Time*,
  el equivalente de `NaN` para fechas) en vez de reventar el programa.

### ⚠️ Error común

Pedirle a `pd.to_datetime()` que adivine el formato de fechas mezcladas sin
indicarle nada (o usando `dayfirst` de forma global) puede hacer que
intercambie día y mes silenciosamente en algunas filas, sin avisar de que
algo ha ido mal — un error mucho más peligroso que uno que rompe el programa,
porque pasa desapercibido. Separar por formato, como arriba, es más trabajo
pero mucho más seguro.

### 💼 Aplicación profesional

Los números con símbolo de moneda y las fechas en formatos distintos son de
los problemas más comunes al recibir datos de fuentes distintas (un Excel
exportado desde España usará `DD/MM/AAAA`, un sistema americano usará
`MM/DD/AAAA`, una base de datos usará ISO). Detectarlo y tratarlo
explícitamente, en vez de confiar en que Pandas "lo adivine bien", es lo que
diferencia un análisis fiable de uno con errores silenciosos.

---

### 📝 Ejercicio 5 (intermedio-avanzado) — Convertir `ingresos_anuales` y `fecha_registro`

Aplica los dos procesos de arriba sobre `clientes_crm` y guarda los
resultados en dos columnas nuevas: `ingresos_limpios` y `fecha_limpia`.
Comprueba con `.dtypes` que ambas tienen el tipo correcto.


In [28]:
# Ejercicio 5 — Tu código aquí


#### 💡 Solución — Ejercicio 5


In [29]:
df_ej5 = clientes_crm.copy()

df_ej5["ingresos_limpios"] = pd.to_numeric(
    df_ej5["ingresos_anuales"].str.replace(r"[^\d]", "", regex=True), errors="coerce"
)

es_iso = df_ej5["fecha_registro"].str.contains("-")
fecha_limpia = pd.Series(pd.NaT, index=df_ej5.index, dtype="datetime64[ns]")
fecha_limpia[es_iso] = pd.to_datetime(df_ej5.loc[es_iso, "fecha_registro"], format="%Y-%m-%d", errors="coerce")
fecha_limpia[~es_iso] = pd.to_datetime(df_ej5.loc[~es_iso, "fecha_registro"], format="%d/%m/%Y", errors="coerce")
df_ej5["fecha_limpia"] = fecha_limpia

print(df_ej5[["ingresos_limpios", "fecha_limpia"]].dtypes)
print(df_ej5[["ingresos_limpios", "fecha_limpia"]])


ingresos_limpios           float64
fecha_limpia        datetime64[ns]
dtype: object


    ingresos_limpios fecha_limpia
0            45000.0   2023-05-12
1            45000.0   2023-05-12
2            38500.0   2023-06-12
3            52000.0   2023-07-01
4                NaN   2023-08-20
5            41000.0   2023-09-15
6            47200.0   2023-10-02
7            39000.0          NaT
8            36000.0   2023-11-20
9            61000.0   2023-12-01
10           41000.0   2023-09-15


## 7. Valores fuera de rango lógico (outliers de negocio)

**Qué es:** valores que son técnicamente números válidos, pero que no tienen
sentido en el mundo real para esa columna concreta — una edad de `200`, un
importe de compra negativo.

### 🎈 Analogía sencilla

Un termómetro que marca `200°C` en el salón de tu casa no está "roto" en el
sentido de que el número no se puede representar — pero cualquiera sabe que
esa lectura no puede ser real. Detectar estos casos es aplicar sentido común
de negocio, no una regla matemática automática.

### 🐍 Código Python


In [30]:
print(clientes_crm[["nombre", "edad"]])


          nombre   edad
0     Ana García   34.0
1     Ana García   34.0
2     Luis Pérez   28.0
3    Eva Martín    -5.0
4      Iván Soto   41.0
5    María López  200.0
6    Carlos Ruiz    NaN
7    Lucía Vidal   31.0
8    Pablo Núñez   27.0
9      Sara Cano   39.0
10   María López   35.0


In [31]:
edad_corregida = clientes_crm["edad"].copy()
fuera_de_rango = (edad_corregida < 0) | (edad_corregida > 100)

print("Edades fuera de rango:", fuera_de_rango.sum())
edad_corregida[fuera_de_rango] = np.nan

print(edad_corregida)


Edades fuera de rango: 2
0     34.0
1     34.0
2     28.0
3      NaN
4     41.0
5      NaN
6      NaN
7     31.0
8     27.0
9     39.0
10    35.0
Name: edad, dtype: float64


In [32]:
# Otra opción: en vez de poner NaN, "recortar" (clip) al límite válido
importe_corregido = clientes_crm["importe_compra"].clip(lower=0)
print(clientes_crm[["nombre", "importe_compra"]].assign(importe_corregido=importe_corregido))


          nombre  importe_compra  importe_corregido
0     Ana García          320.50             320.50
1     Ana García          320.50             320.50
2     Luis Pérez          150.00             150.00
3    Eva Martín            89.99              89.99
4      Iván Soto          410.00             410.00
5    María López          275.30             275.30
6    Carlos Ruiz          -50.00               0.00
7    Lucía Vidal          198.40             198.40
8    Pablo Núñez          320.50             320.50
9      Sara Cano          540.00             540.00
10   María López          275.30             275.30


### 🔍 Explicación línea a línea

- `(edad_corregida < 0) | (edad_corregida > 100)` → una máscara booleana que
  marca como fuera de rango cualquier edad negativa o mayor de 100. El límite
  de `100` es una decisión de negocio, no una regla universal de Pandas.
- Convertir esos valores en `NaN` es la opción más honesta cuando **no
  sabes** cuál era el valor real: mejor admitir que falta el dato, que dejar
  un número que sabes que es mentira.
- `.clip(lower=0)` → en vez de convertir a `NaN`, "recorta" cualquier valor
  por debajo de `0` y lo sube hasta `0` (también admite `upper=` para un
  límite superior). Es una alternativa cuando tiene sentido asumir que el
  valor real estaba simplemente mal capturado, no ausente.

### ⚠️ Error común

Limpiar los outliers **después** de calcular estadísticas como la media o la
desviación estándar — esas estadísticas ya estarían distorsionadas por los
valores imposibles. Siempre: detectar y corregir outliers primero, calcular
métricas de resumen después.

### 💼 Aplicación profesional

No existe una función de Pandas que decida automáticamente qué es un
outlier de negocio — eso requiere conocer el contexto (¿es razonable una
edad de 100 años? ¿y de 90? ¿una compra de 50.000€ es un error o un cliente
corporativo legítimo?). Pandas te da las herramientas (`clip`, máscaras
booleanas, sustitución), pero el criterio lo aporta el analista.

---

### 📝 Ejercicio 6 (avanzado) — Detectar y corregir importes negativos

Sobre `clientes_crm`, cuenta cuántas filas tienen `importe_compra` negativo, y
después corrígelas con `.clip(lower=0)`, guardando el resultado en una
columna nueva `importe_corregido`.


In [33]:
# Ejercicio 6 — Tu código aquí


#### 💡 Solución — Ejercicio 6


In [34]:
negativos = (clientes_crm["importe_compra"] < 0).sum()
print("Importes negativos encontrados:", negativos)

clientes_crm["importe_corregido"] = clientes_crm["importe_compra"].clip(lower=0)
print(clientes_crm[["nombre", "importe_compra", "importe_corregido"]])


Importes negativos encontrados: 1
          nombre  importe_compra  importe_corregido
0     Ana García          320.50             320.50
1     Ana García          320.50             320.50
2     Luis Pérez          150.00             150.00
3    Eva Martín            89.99              89.99
4      Iván Soto          410.00             410.00
5    María López          275.30             275.30
6    Carlos Ruiz          -50.00               0.00
7    Lucía Vidal          198.40             198.40
8    Pablo Núñez          320.50             320.50
9      Sara Cano          540.00             540.00
10   María López          275.30             275.30


## 8. Validación de formato con expresiones regulares

**Qué es:** comprobar que un texto sigue un patrón concreto (como el formato
de un email), no solo que "no esté vacío".

### 🐍 Código Python


In [35]:
patron_email = r"^[\w\.\-]+@[\w\.\-]+\.\w+$"

clientes_crm["email_valido"] = clientes_crm["email"].str.match(patron_email)
print(clientes_crm[["nombre", "email", "email_valido"]])


          nombre                   email  email_valido
0     Ana García   ana.garcia@correo.com          True
1     Ana García   ana.garcia@correo.com          True
2     Luis Pérez   luis.perez@correo.com          True
3    Eva Martín    eva.martin@correo.com          True
4      Iván Soto     ivan_sin_arroba.com         False
5    María López  maria.lopez@correo.com          True
6    Carlos Ruiz  carlos.ruiz@correo.com          True
7    Lucía Vidal                     NaN         False
8    Pablo Núñez  pablo.nunez@correo.com          True
9      Sara Cano    sara.cano@correo.com          True
10   María López  maria.lopez@correo.com          True


### 🔍 Explicación línea a línea (la expresión regular, por piezas)

- `^` y `$` → "desde el principio" y "hasta el final" del texto — exige que
  el patrón cubra el texto completo, no solo un fragmento dentro de él.
- `[\w\.\-]+` → uno o más caracteres que sean letras/números/guion bajo
  (`\w`), puntos o guiones — la parte antes de la `@`.
- `@` → literalmente, el símbolo arroba (obligatorio).
- `[\w\.\-]+\.\w+` → el dominio: algo, un punto, y la extensión
  (`.com`, `.es`...).
- `.str.match(patron)` → comprueba, fila por fila, si el texto completo
  cumple ese patrón, devolviendo `True`/`False`.

### ⚠️ Importante: esto valida **formato**, no que el email exista de verdad

`email_valido = True` solo significa "tiene la forma de un email" — no
garantiza que esa dirección reciba correo de verdad. Para eso haría falta
enviar un email de verificación, algo que Pandas no puede hacer por ti.

### 💼 Aplicación profesional

Validar formatos (emails, códigos postales, números de teléfono, NIFs) con
regex es un primer filtro de calidad de datos muy habitual antes de cargar
datos a un sistema de producción o antes de enviar comunicaciones masivas a
una lista de clientes.

---

### 📝 Ejercicio 7 (avanzado) — Contar emails inválidos

Sobre `clientes_crm`, usa el patrón de arriba para contar cuántos clientes
tienen un email con formato inválido **o** sin email registrado (`NaN`).


In [36]:
# Ejercicio 7 — Tu código aquí


#### 💡 Solución — Ejercicio 7


In [37]:
patron_email = r"^[\w\.\-]+@[\w\.\-]+\.\w+$"
email_valido = clientes_crm["email"].str.match(patron_email)

invalidos_o_vacios = (~email_valido.fillna(False)) | clientes_crm["email"].isna()
print("Emails inválidos o vacíos:", invalidos_o_vacios.sum())
print(clientes_crm.loc[invalidos_o_vacios, ["nombre", "email"]])


Emails inválidos o vacíos: 2
        nombre                email
4    Iván Soto  ivan_sin_arroba.com
7  Lucía Vidal                  NaN


> 💡 `email_valido` puede tener huecos (`NaN`) cuando el email original era
> nulo. `~email_valido.fillna(False)` primero convierte esos huecos en
> `False` (no podemos confirmar que un email vacío sea "válido"), y después
> niega el resultado con `~` para quedarnos con los que fallan.


## 9. Pipeline completo de limpieza, de principio a fin

Vamos a juntar **todo** lo anterior en una secuencia ordenada, con una
función que mide la "salud" del dataset antes y después — el tipo de
pipeline que escribirías en un proyecto real.


In [38]:
def resumen_calidad(df, etiqueta):
    print("---", etiqueta, "---")
    print("Filas:", df.shape[0])
    print("Duplicados exactos:", df.duplicated().sum())
    print("Nulos totales:", int(df.isna().sum().sum()))
    print()


resumen_calidad(clientes_crm, "ANTES DE LIMPIAR")

limpio = clientes_crm.drop_duplicates().copy()

for columna in ["nombre", "ciudad", "producto"]:
    limpio[columna] = limpio[columna].str.strip().str.title()

limpio["ciudad"] = limpio["ciudad"].replace({"Vlencia": "Valencia"})

limpio["ingresos_anuales"] = pd.to_numeric(
    limpio["ingresos_anuales"].str.replace(r"[^\d]", "", regex=True), errors="coerce"
)

es_iso = limpio["fecha_registro"].str.contains("-")
fechas = pd.Series(pd.NaT, index=limpio.index, dtype="datetime64[ns]")
fechas[es_iso] = pd.to_datetime(limpio.loc[es_iso, "fecha_registro"], format="%Y-%m-%d", errors="coerce")
fechas[~es_iso] = pd.to_datetime(limpio.loc[~es_iso, "fecha_registro"], format="%d/%m/%Y", errors="coerce")
limpio["fecha_registro"] = fechas

limpio.loc[(limpio["edad"] < 0) | (limpio["edad"] > 100), "edad"] = np.nan
limpio["importe_compra"] = limpio["importe_compra"].clip(lower=0)

patron_email = r"^[\w\.\-]+@[\w\.\-]+\.\w+$"
limpio["email_valido"] = limpio["email"].str.match(patron_email)

limpio = limpio.sort_values("fecha_registro").drop_duplicates(subset=["id_cliente"], keep="last")

resumen_calidad(limpio, "DESPUÉS DE LIMPIAR")

limpio[["id_cliente", "nombre", "ciudad", "edad", "ingresos_anuales", "fecha_registro", "importe_compra", "email_valido"]]


--- ANTES DE LIMPIAR ---
Filas: 11
Duplicados exactos: 1
Nulos totales: 3



--- DESPUÉS DE LIMPIAR ---
Filas: 9
Duplicados exactos: 0
Nulos totales: 5



,id_cliente,nombre,ciudad,edad,ingresos_anuales,fecha_registro,importe_compra,email_valido
0,1,Ana García,Madrid,34.0,45000.0,2023-05-12,320.50,True
2,2,Luis Pérez,Barcelona,28.0,38500.0,2023-06-12,150.00,True
3,3,Eva Martín,Madrid,NaN,52000.0,2023-07-01,89.99,True
4,4,Iván Soto,Valencia,41.0,NaN,2023-08-20,410.00,False
10,5,María López,Valencia,35.0,41000.0,2023-09-15,275.30,True
6,6,Carlos Ruiz,Sevilla,NaN,47200.0,2023-10-02,0.00,True
8,8,Pablo Núñez,Barcelona,27.0,36000.0,2023-11-20,320.50,True
9,9,Sara Cano,Madrid,39.0,61000.0,2023-12-01,540.00,True
7,7,Lucía Vidal,Sevilla,31.0,39000.0,NaT,198.40,False


### 🔍 Por qué los nulos **suben** después de limpiar (y por qué eso es bueno)

Fíjate en el resumen: probablemente los nulos totales **aumentan** tras la
limpieza, no disminuyen. Esto no es un error — es justo lo que se busca: una
`edad` de `-5` o `200` parecía un dato "completo" antes de limpiar, pero era
una **mentira con apariencia de dato válido**. Al convertirla en `NaN`, el
dataset deja de mentir y empieza a admitir honestamente lo que no sabe. Un
dataset con más `NaN` pero datos reales es más fiable que uno "completo"
lleno de valores inventados o erróneos.

### 💼 Aplicación profesional

Este es el tipo de función (`resumen_calidad`) que conviene escribir una vez
y reutilizar en cada proyecto: te permite demostrar, con números concretos,
el antes y el después de tu trabajo de limpieza — algo muy útil para
justificar ese trabajo ante alguien que solo ve el resultado final.

---

### 🏆 Ejercicio final (nivel pro) — Tu propio pipeline

Sin mirar la celda de arriba, intenta reconstruir tú mismo el pipeline
completo de limpieza sobre `clientes_crm`, pero esta vez:

1. Al deduplicar por `id_cliente`, quédate con la **primera** aparición en
   vez de la última.
2. Además de todo lo anterior, añade una columna `cliente_premium` que sea
   `True` si `ingresos_anuales` (ya limpio) es mayor que `50000`.
3. Al final, imprime cuántos clientes son `premium`.


In [39]:
# Ejercicio Final — Tu código aquí


#### 💡 Solución — Ejercicio Final


In [40]:
limpio2 = clientes_crm.drop_duplicates().copy()

for columna in ["nombre", "ciudad", "producto"]:
    limpio2[columna] = limpio2[columna].str.strip().str.title()

limpio2["ciudad"] = limpio2["ciudad"].replace({"Vlencia": "Valencia"})

limpio2["ingresos_anuales"] = pd.to_numeric(
    limpio2["ingresos_anuales"].str.replace(r"[^\d]", "", regex=True), errors="coerce"
)

es_iso2 = limpio2["fecha_registro"].str.contains("-")
fechas2 = pd.Series(pd.NaT, index=limpio2.index, dtype="datetime64[ns]")
fechas2[es_iso2] = pd.to_datetime(limpio2.loc[es_iso2, "fecha_registro"], format="%Y-%m-%d", errors="coerce")
fechas2[~es_iso2] = pd.to_datetime(limpio2.loc[~es_iso2, "fecha_registro"], format="%d/%m/%Y", errors="coerce")
limpio2["fecha_registro"] = fechas2

limpio2.loc[(limpio2["edad"] < 0) | (limpio2["edad"] > 100), "edad"] = np.nan
limpio2["importe_compra"] = limpio2["importe_compra"].clip(lower=0)

limpio2 = limpio2.sort_values("fecha_registro").drop_duplicates(subset=["id_cliente"], keep="first")

limpio2["cliente_premium"] = limpio2["ingresos_anuales"] > 50000

print("Clientes premium:", limpio2["cliente_premium"].sum())
limpio2[["nombre", "ingresos_anuales", "cliente_premium"]]


Clientes premium: 2


,nombre,ingresos_anuales,cliente_premium
0,Ana García,45000.0,False
2,Luis Pérez,38500.0,False
3,Eva Martín,52000.0,True
4,Iván Soto,NaN,False
5,María López,41000.0,False
6,Carlos Ruiz,47200.0,False
8,Pablo Núñez,36000.0,False
9,Sara Cano,61000.0,True
7,Lucía Vidal,39000.0,False


## 📝 Resumen — Cheat sheet de limpieza

| Problema | Herramienta de Pandas |
|---|---|
| Filas repetidas exactas | `duplicated()`, `drop_duplicates()` |
| Filas repetidas "lógicas" (misma clave, datos distintos) | `duplicated(subset=..., keep=False)` |
| Huecos en los datos | `isna()`, `dropna()`, `fillna()` |
| Texto con espacios/mayúsculas inconsistentes | `.str.strip()`, `.str.title()` / `.str.lower()` |
| Erratas conocidas en categorías | `.replace({"erróneo": "correcto"})` |
| Números guardados como texto con símbolos | `.str.replace(regex)` + `pd.to_numeric(errors="coerce")` |
| Fechas en formatos mixtos | Separar por formato + `pd.to_datetime(format=..., errors="coerce")` |
| Valores fuera de rango lógico | Máscara booleana + `np.nan`, o `.clip()` |
| Formato de texto a validar (email, etc.) | `.str.match(patron_regex)` |

## 🔜 Próximos pasos

Con esto tienes el grueso de lo que se necesita para que un dataset real sea
fiable. El siguiente paso natural es aplicar este mismo criterio sobre datos
reales que tú mismo descargues (un CSV de Kaggle, un export de tu proyecto de
bootcamp), y practicar detectando problemas que no estén ya "puestos a
propósito" como en este notebook — ahí es donde de verdad se afianza esta
habilidad.
